In [2]:
import os
import cv2
import numpy as np
from keras.datasets import mnist
from keras.utils import to_categorical
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.models import load_model

def preprocess_digit(digit_path):
    digit = cv2.imread(digit_path, cv2.IMREAD_GRAYSCALE)
    digit = cv2.resize(digit, (28, 28))
    digit = digit.astype('float32') / 255.0
    digit = np.reshape(digit, (1, 28, 28, 1))
    return digit

def train_model():
    (train_digits, train_labels), (test_digits, test_labels) = mnist.load_data()  # ✅ load test data too

    train_digits = train_digits.reshape((60000, 28, 28, 1)).astype('float32') / 255
    test_digits = test_digits.reshape((10000, 28, 28, 1)).astype('float32') / 255  # ✅ normalize test data

    train_labels = to_categorical(train_labels)
    test_labels = to_categorical(test_labels)  # ✅ one-hot encode test labels

    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D((2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(Flatten())
    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(10, activation='softmax'))

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_digits, train_labels, epochs=10, batch_size=64, validation_split=0.2)

    test_loss, test_acc = model.evaluate(test_digits, test_labels)  # ✅ evaluate and print accuracy
    print("Test Accuracy:", test_acc)

    model.save('mnist_model.h5')
    return model

def classify_digit(digit_path, model):
    input_digit = preprocess_digit(digit_path)
    prediction = model.predict(input_digit)
    predicted_label = np.argmax(prediction)
    return predicted_label

model_path = 'mnist_model.h5'
if not os.path.exists(model_path):
    print("Model not found. Training a new model.")
    model = train_model()
else:
    print("Loading the existing trained model.")
    model = load_model(model_path)

digit_path = '/content/WhatsApp Image 2026-06-07 at 9.01.41 PM (1).jpeg'
if os.path.exists(digit_path):
    predicted_class = classify_digit(digit_path, model)
    print(f'The input digit is predicted to be a {predicted_class}.')

    user_feedback = input("Was the prediction correct? (yes/no): ").lower()
    if user_feedback == 'no':
        print("Retraining the model with the updated data.")
        model = train_model()
        predicted_class = classify_digit(digit_path, model)
        print(f'After retraining, the input digit is predicted to be a {predicted_class}.')
else:
    print(f"Error: digit file not found at {digit_path}.")

Loading the existing trained model.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 114ms/step
The input digit is predicted to be a 4.
Was the prediction correct? (yes/no): no
Retraining the model with the updated data.
Epoch 1/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 46s 58ms/step - accuracy: 0.9140 - loss: 0.2784 - val_accuracy: 0.9784 - val_loss: 0.0724
Epoch 2/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 52s 69ms/step - accuracy: 0.9749 - loss: 0.0838 - val_accuracy: 0.9852 - val_loss: 0.0479
Epoch 3/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 45s 61ms/step - accuracy: 0.9826 - loss: 0.0596 - val_accuracy: 0.9881 - val_loss: 0.0433
Epoch 4/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 81s 59ms/step - accuracy: 0.9864 - loss: 0.0451 - val_accuracy: 0.9885 - val_loss: 0.0389
Epoch 5/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 80s 57ms/step - accuracy: 0.9887 - loss: 0.0388 - val_accuracy: 0.9890 - val_loss: 0.0389
Epoch 6/10
750/750 ━━━━━━━━━━━━━━━━━━━━ 43s 57ms/step - accuracy: 0.9901 - loss: 0.0328 - val_accuracy: 0.9884 - val_loss: 0.0419
Epoch 7/10
750/750 ━

Test Accuracy: 0.9900000095367432
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 95ms/step
After retraining, the input digit is predicted to be a 8.
